__Three-Node MLP Simulations (MLP3)__

Alfred

A 2 --> 3 --> 1 network. The hidden layer is three STAR perceptrons. The output layer is the same
coincidence gate as MLP2, but the sequestering RNA that sets its threshold is no longer made by a
constitutive cassette -- it is the output of the third hidden perceptron, so the threshold itself
is learned.


In [ ]:
# Libraries for mathematical analysis
import numpy as np
from skimage import measure
import sympy as sym
import pylab as pl
import matplotlib.pyplot as plt
import scipy.integrate
import biocircuits

# Libraries to visualize results
import bokeh.io
from bokeh.io import show
from bokeh.plotting import figure
from bokeh.plotting import column
import bokeh.palettes
from bokeh.models import LinearColorMapper, ColorBar
from bokeh.models import Range1d
from bokeh.io import export_svgs
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from bokeh.layouts import row
import re

bokeh.io.output_notebook()

<u>MLP Model</u>

![MLP3 Reaction Scheme](MLP3_Reaction.png)

A two-layer network with **three** STAR perceptrons in the hidden layer ($j$, $k$ and $h$). Each
input plasmid $X_i$ drives both a STAR cassette ($\alpha^{+}$) and a sequestering-RNA cassette
($\alpha^{-}$) at every hidden node, so each effective weight
$w_{n,i} \propto \alpha^{+}_{n,i} - \alpha^{-}_{n,i}$ can take either sign.

__How layer 2 differs from MLP2.__ The gate chemistry is unchanged -- what changes is where the
sequesterer comes from:

| | MLP2 | MLP3 (this notebook) |
|---|---|---|
| Hidden layer | 2 STAR perceptrons ($j$, $k$) | 3 STAR perceptrons ($j$, $k$, $h$) |
| Output complex $C^{(2)}$ | $Y^{(1)}_j \cdot Y^{(1)}_k$ | $Y^{(1)}_j \cdot Y^{(1)}_k$ (same) |
| Sequesterer of $Y^{(1)}_k$ | $Z^{(2)}$, made constitutively at $\theta$ | $Y^{(1)}_h$, the third perceptron's output |
| Threshold | fixed, set by $\theta / \phi$ | **input-dependent**, set by $w_h \cdot x$ |
| Free parameters at the gate | one scalar $\theta$ | a whole weight row $(\alpha^{+}_{h,i}, \alpha^{-}_{h,i})$ |

There is no $\theta$ and no $Z^{(2)}$ anywhere in this scheme. The constitutive cassette of MLP2 has
been replaced by a trainable perceptron, so the amount of $Y^{(1)}_k$ held off the gate now depends
on the inputs instead of being a constant. This is the one structural change, and everything
interesting about MLP3 follows from it.

__Roles of the three hidden nodes.__ They are not interchangeable:
-   $j$ and $k$ are the two halves of the output complex -- both must be ON for any output, exactly
    as in MLP2. The gate is a product, so either one alone gives zero.
-   $h$ is purely **inhibitory**. Its transcript never reaches $C^{(2)}$; it only binds $Y^{(1)}_k$
    into the inert complex $Q$. Raising $h$ can only lower the output.

__Wiring between the layers.__ As in MLP1 and MLP2, the released transcripts of layer 1 *are* the
input species of layer 2 -- they are not re-transcribed from a plasmid. Writing it this way makes
the downstream loading/retroactivity terms appear automatically and avoids double counting.

__Layer 1 -- STAR perceptrons__ ($n \in \{j, k, h\}$, target $P^{(1)}_n$ is catalytic and conserved):
\begin{align*}
&\dot{m}^{(1)}_n = \textstyle\sum_i \alpha^{+}_{n,i} x_i - \phi m^{(1)}_n - a^{(1)}_n p^{(1)}_n m^{(1)}_n + d^{(1)}_n c^{(1)}_n - b^{(1)}_n z^{(1)}_n m^{(1)}_n + u^{(1)}_n q^{(1)}_n\\
&\dot{z}^{(1)}_n = \textstyle\sum_i \alpha^{-}_{n,i} x_i - \phi z^{(1)}_n - b^{(1)}_n z^{(1)}_n m^{(1)}_n + u^{(1)}_n q^{(1)}_n\\
&\dot{p}^{(1)}_n = d^{(1)}_n c^{(1)}_n - a^{(1)}_n p^{(1)}_n m^{(1)}_n + v^{(1)}_n c^{(1)}_n\\
&\dot{c}^{(1)}_n = a^{(1)}_n p^{(1)}_n m^{(1)}_n - d^{(1)}_n c^{(1)}_n - v^{(1)}_n c^{(1)}_n\\
&\dot{q}^{(1)}_n = b^{(1)}_n z^{(1)}_n m^{(1)}_n - u^{(1)}_n q^{(1)}_n - \phi q^{(1)}_n
\end{align*}

__Layer-1 outputs (= layer-2 inputs).__ These carry the extra terms for being consumed downstream.
$y^{(1)}_j$ and $y^{(1)}_k$ are consumed by the $a_2$ gate reaction; $y^{(1)}_k$ and $y^{(1)}_h$ are
consumed by the $b_2$ sequestration reaction, so $y^{(1)}_k$ carries both:
\begin{align*}
&\dot{y}^{(1)}_j = v^{(1)}_j c^{(1)}_j - \phi y^{(1)}_j - a_2 y^{(1)}_j y^{(1)}_k + d_2 c^{(2)}\\
&\dot{y}^{(1)}_k = v^{(1)}_k c^{(1)}_k - \phi y^{(1)}_k - a_2 y^{(1)}_j y^{(1)}_k + d_2 c^{(2)} - b_2 y^{(1)}_k y^{(1)}_h + u_2 q^{(2)}\\
&\dot{y}^{(1)}_h = v^{(1)}_h c^{(1)}_h - \phi y^{(1)}_h - b_2 y^{(1)}_k y^{(1)}_h + u_2 q^{(2)}
\end{align*}

__Layer 2 -- coincidence node with a learned threshold__:
\begin{align*}
&\dot{c}^{(2)} = a_2 y^{(1)}_j y^{(1)}_k - d_2 c^{(2)} - \phi c^{(2)} - v_2 c^{(2)}\\
&\dot{y}^{(2)} = v_2 c^{(2)} - \delta_2 y^{(2)}\\
&\dot{q}^{(2)} = b_2 y^{(1)}_k y^{(1)}_h - u_2 q^{(2)} - \phi q^{(2)}
\end{align*}

Where:
-   $X_1, X_2 = \text{Input plasmids (each carrying an } \alpha^{+} \text{ and an } \alpha^{-} \text{ cassette per hidden node)}$
-   $M^{(1)}_n = \text{STAR for hidden node } n$
-   $Z^{(1)}_n = \text{Sequestering RNA for hidden node } n$
-   $P^{(1)}_n = \text{Target RNA (paused elongation complex) of hidden node } n$
-   $C^{(1)}_n = \text{STAR-Target complex of hidden node } n$
-   $Y^{(1)}_n = \text{Released transcript of hidden node } n \text{ (= a layer-2 input)}$
-   $Q^{(1)}_n = \text{Sequestered STAR of hidden node } n$
-   $C^{(2)} = \text{Output complex } (Y^{(1)}_j \cdot Y^{(1)}_k)$
-   $Y^{(2)} = \text{Network output}$
-   $Q^{(2)} = \text{Sequestered layer-2 input } (Y^{(1)}_k \cdot Y^{(1)}_h)$


<u>Simulated Decision Boundaries & Activation Functions</u>

In [ ]:
#Function to help calculate the ODEs
#X --> Sets all concentrations to 0 at the start, except the layer-1 targets (conserved)
def mlpODEs(X, t, x1, x2, rates):

    #Layer 1 nodes j, k, h, then layer 2
    (m_j, z_j, p_j, c_j, y_j, q_j,
     m_k, z_k, p_k, c_k, y_k, q_k,
     m_h, z_h, p_h, c_h, y_h, q_h,
     c_2, y_2, q_2) = X

    #Unpack rates from array
    alphaPos = rates["alphaPos"]  #Transcription of STAR per unit input, [node, input]
    alphaNeg = rates["alphaNeg"]  #Transcription of sequestering RNA per unit input, [node, input]
    phi = rates["phi"]          #Decay of RNA (shared)

    a_j = rates["a_j"]          #Binding of STAR & target, node j
    d_j = rates["d_j"]          #Dissociation of STAR & target, node j
    v_j = rates["v_j"]          #Release of transcript, node j
    b_j = rates["b_j"]          #Binding of STAR & sequesterer, node j
    u_j = rates["u_j"]          #Dissociation of STAR & sequesterer, node j

    a_k = rates["a_k"]          #Same, for node k
    d_k = rates["d_k"]
    v_k = rates["v_k"]
    b_k = rates["b_k"]
    u_k = rates["u_k"]

    a_h = rates["a_h"]          #Same, for node h --> the inhibitory node
    d_h = rates["d_h"]
    v_h = rates["v_h"]
    b_h = rates["b_h"]
    u_h = rates["u_h"]

    a_2     = rates["a_2"]      #Binding of y_j & y_k to each other --> forms the output complex
    d_2     = rates["d_2"]      #Dissociation
    v_2     = rates["v_2"]      #Conversion of the output complex to output
    delta_2 = rates["delta_2"]  #Decay of output RNA
    b_2     = rates["b_2"]      #Binding of y_k & y_h --> the learned threshold
    u_2     = rates["u_2"]      #Dissociation

    #Weight-summed transcription into each hidden node --> alphaPos/alphaNeg are [node, input]
    xin   = np.array([x1, x2])
    prodM = alphaPos @ xin
    prodZ = alphaNeg @ xin

    #LAYER 1, NODE j --> STAR kinetics, target is catalytic so p_j + c_j is conserved
    dm_j = prodM[0] - phi*m_j - a_j*p_j*m_j + d_j*c_j - b_j*z_j*m_j + u_j*q_j
    dz_j = prodZ[0] - phi*z_j - b_j*z_j*m_j + u_j*q_j
    dp_j = d_j*c_j - a_j*p_j*m_j + v_j*c_j
    dc_j = a_j*p_j*m_j - d_j*c_j - v_j*c_j
    dq_j = b_j*z_j*m_j - u_j*q_j - phi*q_j

    #LAYER 1, NODE k --> the contested species: it feeds the gate AND is sequestered by node h
    dm_k = prodM[1] - phi*m_k - a_k*p_k*m_k + d_k*c_k - b_k*z_k*m_k + u_k*q_k
    dz_k = prodZ[1] - phi*z_k - b_k*z_k*m_k + u_k*q_k
    dp_k = d_k*c_k - a_k*p_k*m_k + v_k*c_k
    dc_k = a_k*p_k*m_k - d_k*c_k - v_k*c_k
    dq_k = b_k*z_k*m_k - u_k*q_k - phi*q_k

    #LAYER 1, NODE h --> the threshold node. Its transcript never reaches C2, it only sequesters y_k
    dm_h = prodM[2] - phi*m_h - a_h*p_h*m_h + d_h*c_h - b_h*z_h*m_h + u_h*q_h
    dz_h = prodZ[2] - phi*z_h - b_h*z_h*m_h + u_h*q_h
    dp_h = d_h*c_h - a_h*p_h*m_h + v_h*c_h
    dc_h = a_h*p_h*m_h - d_h*c_h - v_h*c_h
    dq_h = b_h*z_h*m_h - u_h*q_h - phi*q_h

    #LAYER-1 OUTPUTS == LAYER-2 INPUTS
    #y_j and y_k bind EACH OTHER to make the output complex --> the a_2 term is shared
    dy_j = v_j*c_j - phi*y_j - a_2*y_j*y_k + d_2*c_2
    #y_k is consumed by the gate reaction AND by sequestration with y_h
    dy_k = v_k*c_k - phi*y_k - a_2*y_j*y_k + d_2*c_2 - b_2*y_k*y_h + u_2*q_2
    #y_h is consumed only by the sequestration reaction --> it never contributes to the output
    dy_h = v_h*c_h - phi*y_h - b_2*y_k*y_h + u_2*q_2

    #LAYER 2 --> coincidence node, not catalytic (neither gate input is regenerated)
    dc_2 = a_2*y_j*y_k - d_2*c_2 - phi*c_2 - v_2*c_2
    dy_2 = v_2*c_2 - delta_2*y_2
    dq_2 = b_2*y_k*y_h - u_2*q_2 - phi*q_2

    return [dm_j, dz_j, dp_j, dc_j, dy_j, dq_j,
            dm_k, dz_k, dp_k, dc_k, dy_k, dq_k,
            dm_h, dz_h, dp_h, dc_h, dy_h, dq_h,
            dc_2, dy_2, dq_2]


#Index of the network output y_2 in the state vector --> Used everywhere downstream
OUT = 19

#Indices of the three layer-1 targets --> Used by the sanity checks to disable a node
P_IDX = {"j": 2, "k": 8, "h": 14}

#Indices of the three layer-1 released transcripts
Y_IDX = {"j": 4, "k": 10, "h": 16}

Layer 1 ($n \in \{j,k,h\}$, STAR -- the target is catalytic so there is no $\phi$ on $c$):

$K_{C}^{(1)} = \frac{v^{(1)}_n + d^{(1)}_n}{a^{(1)}_n}\\$

$K_{Q}^{(1)} = \frac{\phi + u^{(1)}_n}{b^{(1)}_n}$

Layer 2 (coincidence node -- the output complex decays, so $\phi$ appears in $K_C$):

$K_{C}^{(2)} = \frac{\phi + v_2 + d_2}{a_2}\\$

$K_{Q}^{(2)} = \frac{\phi + u_2}{b_2}$

Unlike MLP2, there is **no** $\theta/\phi$ ceiling on the sequesterer pool. The amount of
$Y^{(1)}_h$ available to hold off $Y^{(1)}_k$ is whatever the third perceptron transcribes, so the
threshold scale is set by node $h$'s own weights and release rate:

$$y^{(1)}_h \sim \frac{v^{(1)}_h}{\phi} \cdot f\!\left(\textstyle\sum_i (\alpha^{+}_{h,i} - \alpha^{-}_{h,i}) x_i\right)$$

That is the whole point of the architecture: in MLP2 this quantity was a constant, here it is a
function of the input.


In [ ]:
# Define a grid of input values (x1, x2)
x1 = np.linspace(0, 1, 50)
x2 = np.linspace(0, 1, 50)
X1, X2 = np.meshgrid(x1, x2)

#Activation function inputs
x1Vals = np.linspace(0, 1, 50)
x2Vals = [0.5, 0.5, 0.5]

# Simulation time --> long enough for the 21-species system to reach steady state
t = np.linspace(0, 300, 600)

#Total layer-1 target supply --> p + c is conserved at this value for each hidden node
#It is an initial condition because that is the only place a conserved quantity can live
pTot1 = 0.5

#Initial species states --> layer-1 targets start at pTot1 because they are conserved
#There is no z_2 in MLP3: the sequesterer is y_h, which node h transcribes for itself
X0 = [0,      #m_j
      0,      #z_j
      0.5,    #p_j --> P is never produced in STAR --> Needs to be given
      0,      #c_j
      0,      #y_j
      0,      #q_j
      0,      #m_k
      0,      #z_k
      0.5,    #p_k --> P is never produced in STAR --> Needs to be given
      0,      #c_k
      0,      #y_k
      0,      #q_k
      0,      #m_h
      0,      #z_h
      0.5,    #p_h --> P is never produced in STAR --> Needs to be given
      0,      #c_h
      0,      #y_h
      0,      #q_h
      0,      #c_2
      0,      #y_2
      0]      #q_2

#Extra input weights --> Production terms become alphaPos @ (w1*X1, w2*X2)
w1 = 1
w2 = 1

#Kinetics parameters --> each variant is a different operating regime
#alphaPos/alphaNeg are stacked per variant --> each entry is a [node, input] matrix with THREE
#rows now: j and k are the gate partners, h is the inhibitory threshold node

                      #X1     X2
alphaPos1 = np.array([[ 2.00,  0.20],   #node j
                      [ 0.20,  2.00],   #node k
                      [ 1.00,  1.00]])   #node h
                      #X1     X2
alphaNeg1 = np.array([[ 0.20,  0.80],   #node j
                      [ 0.80,  0.20],   #node k
                      [ 0.20,  0.20]])   #node h

                      #X1     X2
alphaPos2 = np.array([[ 2.00,  0.20],   #node j
                      [ 0.20,  2.00],   #node k
                      [ 1.60,  0.40]])   #node h
                      #X1     X2
alphaNeg2 = np.array([[ 0.20,  0.80],   #node j
                      [ 0.80,  0.20],   #node k
                      [ 0.20,  0.20]])   #node h

                      #X1     X2
alphaPos3 = np.array([[ 2.50,  0.30],   #node j
                      [ 0.30,  2.50],   #node k
                      [ 0.60,  1.80]])   #node h
                      #X1     X2
alphaNeg3 = np.array([[ 0.30,  0.60],   #node j
                      [ 0.60,  0.30],   #node k
                      [ 0.20,  0.20]])   #node h

                      #X1     X2
alphaPos4 = np.array([[ 1.60,  0.40],   #node j
                      [ 0.40,  1.60],   #node k
                      [ 1.20,  1.20]])   #node h
                      #X1     X2
alphaNeg4 = np.array([[ 0.20,  0.50],   #node j
                      [ 0.50,  0.20],   #node k
                      [ 0.30,  0.30]])   #node h

                      #X1     X2
alphaPos5 = np.array([[ 3.00,  0.20],   #node j
                      [ 0.20,  2.20],   #node k
                      [ 0.90,  0.90]])   #node h
                      #X1     X2
alphaNeg5 = np.array([[ 0.20,  1.00],   #node j
                      [ 1.00,  0.20],   #node k
                      [ 0.20,  0.20]])   #node h

ratesDefault = {
    'alphaPos': np.array([alphaPos1, alphaPos2, alphaPos3, alphaPos4, alphaPos5]),
    'alphaNeg': np.array([alphaNeg1, alphaNeg2, alphaNeg3, alphaNeg4, alphaNeg5]),
    'phi':         np.array([     0.1,      0.1,      0.2,      0.1,     0.07]),
    'a_j':         np.array([      10,       25,       10,       10,       10]),
    'd_j':         np.array([       1,      0.5,        1,        1,        1]),
    'v_j':         np.array([       1,        1,        2,        1,        1]),
    'b_j':         np.array([     200,      200,      200,      200,      400]),
    'u_j':         np.array([     0.2,      0.2,      0.2,      0.2,     0.05]),
    'a_k':         np.array([      10,       10,       30,       10,        5]),
    'd_k':         np.array([       1,        1,        1,      2.5,        1]),
    'v_k':         np.array([       1,        1,        1,        1,        1]),
    'b_k':         np.array([     200,      400,      200,      200,      200]),
    'u_k':         np.array([     0.2,      0.1,      0.2,      0.2,      0.2]),
    'a_h':         np.array([      10,       10,       10,       30,       10]),
    'd_h':         np.array([       1,        1,        1,        1,      0.4]),
    'v_h':         np.array([       1,        1,        1,      1.5,        1]),
    'b_h':         np.array([     200,      200,       80,      200,      200]),
    'u_h':         np.array([     0.2,      0.2,      0.2,      0.2,      0.2]),
    'a_2':         np.array([      10,       10,       25,       10,       40]),
    'd_2':         np.array([       1,        1,        1,        1,        1]),
    'v_2':         np.array([       1,        1,        2,        1,      1.5]),
    'delta_2':     np.array([     0.1,      0.1,      0.1,     0.05,      0.1]),
    'b_2':         np.array([      50,      120,       50,      200,       80]),
    'u_2':         np.array([     0.2,      0.2,      0.2,     0.05,      0.2]),
}

#Variant used by the sanity checks, the node-h sweeps and the separability search
CHECK_VARIANT = 0


#getVariantRates() --> Extracts variant-specific rates from ratesDefault
def getVariantRates(ratesArray, variantIndex):
    rates = {}
    for key, arr in ratesArray.items():
        rates[key] = arr[variantIndex]
    return rates


#scaleNodeH() --> Copy of a [node, input] weight matrix with ONLY node h's row scaled.
#This is MLP3's analogue of changing theta in MLP2, so it gets its own helper.
def scaleNodeH(alphaMatrix, scale):
    scaled = np.array(alphaMatrix, dtype=float).copy()
    scaled[2] = scaled[2] * scale
    return scaled


#getVariantLabel() --> Legend label (labelBy = "alpha", "wh", "xi2", "K_C2", "K_Q2", or "None")
def getVariantLabel(rates, labelBy):
    if labelBy in ("None", None):
        return None
    if labelBy == "alpha":
        #The two matrices side by side are too wide for a legend --> show the net drive they produce
        netDrive = rates["alphaPos"] - rates["alphaNeg"]
        return "W = " + np.array2string(netDrive, precision=2, separator=" ").replace("\n", "")
    if labelBy == "wh":  #Net drive of the threshold node --> MLP3's replacement for "theta"
        wh = rates["alphaPos"][2] - rates["alphaNeg"][2]
        return f"w_h = [{wh[0]:+.2f}, {wh[1]:+.2f}]"
    if labelBy == "xi2":  # xi = K_Q2 / K_C2 = a_2(phi + u_2) / (b_2(phi + v_2 + d_2))
        return (f"ξ2 = {rates['a_2'] * (rates['phi'] + rates['u_2']) / (rates['b_2'] * (rates['phi'] + rates['v_2'] + rates['d_2'])):.3g}")
    if labelBy == "K_C2":
        return f"K_C2 = {(rates['phi'] + rates['v_2'] + rates['d_2']) / rates['a_2']:.3g}"
    if labelBy == "K_Q2":
        return f"K_Q2 = {(rates['phi'] + rates['u_2']) / rates['b_2']:.3g}"
    return None


numVariants = len(ratesDefault['phi']) #phi is in all models --> Best choice

#Bokeh color palette for heatmaps + activation functions
colors = bokeh.palettes.OrRd3

In [ ]:
# Function to do the 2D projection of decision boundaries with normalization  <-- From the EBCP LATAM 2025 Workshop
def contourf(p, x, y, z, title=None, palette="Spectral11", normal = True):
    """Make a filled contour plot given x, y, z data given in 2D arrays."""

    # Normalize the z values
    if normal:
      z_min = z.min()
      z_max = z.max()
      z_normalized = (z - z_min) / (z_max - z_min)  # Normalize to range [0, 1]
    else:
      z_normalized = z

    # Add zero padding at the boundaries for better visualization
    N = z_normalized.shape[1]
    z0 = np.c_[z_normalized, np.zeros(N)]
    z0[-1, -1] = 1.  # Ensure the padding contains a value within range

    # Plot the normalized values
    p.image(
        image=[z0],
        x=x.min(),
        y=y.min(),
        dw=(x.max() - x.min()) * (1 + 1 / N),
        dh=x.max() - x.min(),
        palette=palette,
        alpha=0.8,
    )

    # Color mapping based on the normalized z0 values
    color = LinearColorMapper(palette=palette, low=z0.min(), high=z0.max())
    cb = ColorBar(color_mapper=color, location=(0, 0), width=10)
    p.add_layout(cb, 'right')

    return ()

In [ ]:
#Array to store all results
results = []

#Range through each variant --> Solve ODEs + Plot heatmap + Plot activation function --> Append to Results
#NOTE: 50x50 grid x 5 variants x a 21-species system --> this cell takes several minutes
for variantIndex in range(numVariants):
    #Get rates for this current iteration's variant
    rates = getVariantRates(ratesDefault, variantIndex)

    #ODE SOLUTIONS
    #Array for storing scipy.integrate solutions
    gridSolutions = [[None for _ in range(len(x1))] for _ in range(len(x2))]

    # Initialize a matrix to store the steady-state output y
    Y = np.zeros_like(X1)

    # Iterate over the grid values for all variable inputs X1 and X2 --> Solve the ODEs
    for i in range(len(x1)):
        for j in range(len(x2)):
            solutions = scipy.integrate.odeint(mlpODEs, X0, t, args=(w1 * X1[j, i], w2 * X2[j, i], rates))
            gridSolutions[j][i] = solutions
            Y[j, i] = solutions[-1, OUT] #Extract the steady state output y_2

    #HEATMAPS
    p_heatmap = figure(width=350, height=300, title=f"Variant {variantIndex}")

    #Make the heatmap
    contourf(
        p_heatmap,
        X1,
        X2,
        Y,
        palette=bokeh.palettes.Oranges8[::-1],
        normal=True
    )

    p_heatmap.xaxis.axis_label = "X1"
    p_heatmap.yaxis.axis_label = "X2"

    #ACTIVATION FUNCTIONS
    activationFunctions = {}
    p_activation = figure(width=350, height=300, title=f"Variant {variantIndex}")

    for x2j, color in zip(x2Vals, colors):

        curves = []
        output_y = np.zeros_like(x1Vals)

        for i, x1i in enumerate(x1Vals):

            sol = scipy.integrate.odeint(
                mlpODEs, X0, t,
                args=(w1 * x1i, w2 * x2j, rates)
            )

            curves.append(sol)
            output_y[i] = sol[-1, OUT]

        activationFunctions[x2j] = curves

        p_activation.line(
            x1Vals,
            output_y,
            line_width=3,
            color=color,
            legend_label=f"X2 = {x2j}"
        )

    p_activation.xaxis.axis_label = "X1 (Input)"
    p_activation.yaxis.axis_label = "y_2 (Network Output)"
    p_activation.legend.location = "top_left"

    #Append ODE solutions, heatmaps, and activation functions for all variants into the results array
    results.append({
        "variantIndex": variantIndex,
        "rates": rates,
        "solutions": {
            "grid": gridSolutions,
            "activation": activationFunctions
        },
        "heatmaps": p_heatmap,
        "activationFunctions": p_activation
    })

#Plot results
for res in results:
    print(f"\nVariant {res['variantIndex']}")
    for k, v in res["rates"].items():
        print(f"{k}: {v}")

    show(row(res["heatmaps"], res["activationFunctions"]))

In [ ]:
###Sanity checks --> run after the variant loop. Every number here is an invariant of the
###model, not a fitted result, so a failure means a term was mistyped rather than mis-tuned.
checkRates = getVariantRates(ratesDefault, CHECK_VARIANT)
sol = scipy.integrate.odeint(mlpODEs, X0, t, args=(w1 * 1.0, w2 * 1.0, checkRates))
final = sol[-1]

#1. Steady state --> the grid results are read off the last time point, so it must be converged
deriv = np.array(mlpODEs(final, t[-1], w1 * 1.0, w2 * 1.0, checkRates))
print(f"1. max |dX/dt| at t = {t[-1]:g}:  {np.abs(deriv).max():.3g}   (should be ~0)")

#2. Layer-1 target conservation --> p + c has no production or decay term, so it is pinned to pTot1
#   This catches an accidental phi on a layer-1 target
for n in ("j", "k", "h"):
    ip = P_IDX[n]
    print(f"2. node {n}  p + c = {final[ip] + final[ip + 1]:.6f}   (pTot1 = {pTot1})")

#3. j and k are the gate partners --> killing EITHER must kill the output.
#   h is purely inhibitory --> killing it must RAISE the output instead.
#   This is the structural signature of MLP3 and the one check MLP2 cannot pass.
yBase = final[OUT]
print(f"3. baseline y_2 at (1, 1) = {yBase:.6f}")
for n in ("j", "k", "h"):
    X0killed = list(X0)
    X0killed[P_IDX[n]] = 0.0   #No target supply --> that node can never release y
    yKilled = scipy.integrate.odeint(mlpODEs, X0killed, t, args=(w1 * 1.0, w2 * 1.0, checkRates))[-1, OUT]
    expected = "should be 0" if n in ("j", "k") else f"should EXCEED {yBase:.4f}"
    print(f"   node {n} disabled (p_{n}(0) = 0):  y_2 = {yKilled:.6f}   ({expected})")

#4. Node h is a threshold, not a bias --> scaling up its drive can only lower the output, and it
#   can never lift the output off zero at the origin
print("4. node h drive sweep at (x1, x2) = (1, 1), and at the origin:")
for s in (0.0, 0.25, 0.5, 1.0, 2.0, 4.0):
    r = dict(checkRates)
    r["alphaPos"] = scaleNodeH(checkRates["alphaPos"], s)
    yOn  = scipy.integrate.odeint(mlpODEs, X0, t, args=(w1 * 1.0, w2 * 1.0, r))[-1, OUT]
    yOff = scipy.integrate.odeint(mlpODEs, X0, t, args=(0.0, 0.0, r))[-1, OUT]
    print(f"   h drive x{s:<5g}  y_2(1,1) = {yOn:.4f}   y_2(0,0) = {yOff:.4f}")

#5. Mass is conserved through the sequestration reaction --> every y_h that disappears is either
#   free, bound in q_2, or diluted. A quick consistency check on the q_2 balance at steady state:
print(f"5. q_2 balance:  b_2*y_k*y_h = {checkRates['b_2'] * final[Y_IDX['k']] * final[Y_IDX['h']]:.6g}"
      f"   vs   (u_2 + phi)*q_2 = {(checkRates['u_2'] + checkRates['phi']) * final[20]:.6g}")

In [ ]:
### plotActivationFunctionVariants() --> Plots the network activation function for every rate variant on the same figure, holding X2 fixed, so the effect of the varied rate(s) is directly comparable
def plotActivationFunctionVariants(ratesDefault, x2_fixed, x1Vals, w1, w2, X0, t, labelBy="None", colors=None):
    numVariants = len(ratesDefault['phi'])  # phi is in all models --> Best choice
    if colors is None:
        colors = bokeh.palettes.OrRd3 if numVariants <= 3 else bokeh.palettes.Category10[10]

    p_activation = figure(width=500, height=400, x_range=(0, 1), title=f"Network Activation Function Across Rate Variants")

    for variantIndex in range(numVariants):
        rates = getVariantRates(ratesDefault, variantIndex)

        output_y = np.zeros_like(x1Vals)
        for i, x1i in enumerate(x1Vals):
            sol = scipy.integrate.odeint(
                mlpODEs, X0, t,
                args=(w1 * x1i, w2 * x2_fixed, rates)
            )
            output_y[i] = sol[-1, OUT]  # Steady-state output y_2

        line_kwargs = {"line_width": 3, "color": colors[variantIndex]}
        label = getVariantLabel(rates, labelBy)
        if label is not None:
            line_kwargs["legend_label"] = label  #labelBy="None" --> No legend

        p_activation.line(x1Vals, output_y, **line_kwargs)

    p_activation.xaxis.axis_label = "X1 (Input)"
    p_activation.yaxis.axis_label = "y_2 (Network Output)"
    if p_activation.legend:
        p_activation.legend.location = "top_left"

    show(p_activation)
    return p_activation

In [ ]:
### Example call --> Hold X2 fixed at one of the existing x2Vals
p1 = plotActivationFunctionVariants(ratesDefault, x2_fixed=x2Vals[0], x1Vals=x1Vals, w1=w1, w2=w2, X0=X0, t=t, labelBy="wh")

#Save plot
#p1.output_backend = "svg"
#export_svgs(p1, filename="Simulation_Plots/mlp3/mlp3_activation_variants.svg")

In [ ]:
###plotDecisionBoundaryVariants() --> Extracts the Y = threshold contour (the decision boundary) for each rate variant and overlays them on one (X1, X2) plot
def plotDecisionBoundaryVariants(ratesDefault, threshold, x1, x2, X1, X2, w1, w2, X0, t, relative=False, labelBy="None", colors=None):
    numVariants = len(ratesDefault['phi'])  # phi is in all models --> Best choice
    if colors is None:
        colors = bokeh.palettes.OrRd3 if numVariants <= 3 else bokeh.palettes.Category10[10]  #OrRd3 only has 3 colors --> Use Category10 for more variants

    p_boundary = figure(width=500, height=400, x_range=(0, 1), y_range=(0, 1), title=f"Decision Boundary Across Rate Variants (Y = {threshold}{' * max(Y)' if relative==True else ''})")

    for variantIndex in range(numVariants):
        rates = getVariantRates(ratesDefault, variantIndex)

        # Solve the ODEs over the full (X1, X2) grid to get steady-state output Y
        Y = np.zeros_like(X1)
        for i in range(len(x1)):
            for j in range(len(x2)):
                sol = scipy.integrate.odeint(mlpODEs, X0, t, args=(w1 * X1[j, i], w2 * X2[j, i], rates))
                Y[j, i]  = sol[-1, OUT]  # Steady-state output y_2

        # relative=True --> threshold is a fraction of this variant's max Y
        level = threshold * Y.max() if relative else threshold

        # Extract the Y = level contour --> This is the decision boundary
        contours = measure.find_contours(Y, level=level)

        if len(contours) == 0:
            print(f"Variant {variantIndex}: Y never crosses {level:.3g} on this grid (max Y = {Y.max():.3g}) --> no boundary to plot")
            continue

        #A contour of only a few points is a degenerate blob (usually a threshold set just above
        #Y = 0 at the origin) --> it draws as nothing, so say so rather than failing silently
        longest = max(len(c) for c in contours)
        if longest < 5:
            print(f"Variant {variantIndex}: boundary at {level:.3g} is degenerate ({longest} points) --> raise the threshold")

        for c, contour in enumerate(contours):
            #find_contours returns [row_idx, col_idx] into the Y grid --> map indices back to real x1, x2 values
            x1_contour = np.interp(contour[:, 1], np.arange(len(x1)), x1)
            x2_contour = np.interp(contour[:, 0], np.arange(len(x2)), x2)

            line_kwargs = {"line_width": 3, "color": colors[variantIndex]}
            if c == 0 and labelBy not in ("None", None):
                line_kwargs["legend_label"] = getVariantLabel(rates, labelBy)  #Only label once per variant

            p_boundary.line(x1_contour, x2_contour, **line_kwargs)

    p_boundary.xaxis.axis_label = "X1 (Input plasmid 1)"
    p_boundary.yaxis.axis_label = "X2 (Input plasmid 2)"
    if p_boundary.legend:
        p_boundary.legend.location = "top_left"

    show(p_boundary)
    return p_boundary

In [ ]:
###threshold = Y value (fraction of max Y when relative=True) --> HELD CONSTANT
###Use relative=True with 0.3-0.5: an absolute threshold near 0 picks out only the origin cell
p2 = plotDecisionBoundaryVariants(ratesDefault, threshold=0.5, x1=x1, x2=x2, X1=X1, X2=X2, w1=w1, w2=w2, X0=X0, t=t, relative=True, labelBy="wh")

#Save plot
#p2.output_backend = "svg"
#export_svgs(p2, filename="Simulation_Plots/mlp3/mlp3_decision_boundary_variants.svg")

<u>The Learned Threshold -- MLP3's One New Knob</u>

This is the section that distinguishes MLP3 from MLP2. In MLP2 the layer-2 threshold was the single
scalar $\theta$. Here it is $y^{(1)}_h$, the output of a perceptron with its own weight row, so the
threshold has a **magnitude** *and* a **direction**. Two sweeps separate those:

-   **Scaling node $h$'s drive** ($\alpha^{+}_{h,\cdot} \to s \cdot \alpha^{+}_{h,\cdot}$) is the closest
    analogue of MLP2's $\theta$ sweep.
-   **Rotating node $h$'s weight row** at fixed magnitude changes *which* inputs raise the
    threshold. MLP2 has no equivalent -- a constitutive cassette has no direction.

__Two honest caveats, both visible in the cells below.__

__1. The magnitude sweep saturates.__ Scaling $h$'s drive suppresses the output hard at first and
then stops mattering: once $y^{(1)}_h$ comfortably exceeds $y^{(1)}_k$, essentially all of
$y^{(1)}_k$ is already held in $Q$ and more $y^{(1)}_h$ changes nothing. In the baseline variant the
output at $(1,1)$ falls $4.15 \to 3.47 \to 1.72$ as $s$ goes $0 \to 0.25 \to 0.5$, then flattens
($1.57$ at $s=1$, $1.55$ at $s=2$). So this knob behaves like a saturating gain on the output
amplitude. Because the boundary plots draw a contour at a *fraction of each variant's own maximum*,
that amplitude change is largely normalized away -- the effect is much clearer in the activation
plot, which is in absolute units.

__2. Rotation only matters when the ON region is asymmetric.__ This is the more interesting
constraint and it is structural, not a tuning artifact. With $j$ driven by $X_1$ and $k$ by $X_2$,
the ON region is a symmetric corner at $(1,1)$ -- and near that corner $x_1 \approx x_2$, where
*every* direction of $w_h$ delivers the same drive ($\cos\psi + \sin\psi$ is $1$ at both $0°$ and
$90°$). Rotating $h$ therefore does almost nothing: rotating from $0°$ to $90°$ re-labels well under
$10\%$ of the input square.

Make the ON region a **band** instead -- node $j$ driven broadly by both inputs, node $k$ requiring
$X_2$ -- and the same rotation re-labels roughly $45\%$ of it, with the ON area more than doubling.
The diagnostic cell further below prints both figures for the grid actually used, so the comparison
is checkable rather than asserted. Both cases are plotted, because the contrast is the lesson: node
$h$ buys a
*directional* threshold, but that direction is only expressive against a hidden layer whose own ON
region is not symmetric about the diagonal.


In [ ]:
###Node-h drive sweep --> MLP3's analogue of MLP2's theta sweep.
#Everything except node h's positive weight row is held at the baseline variant.
hScales = [0.0, 0.25, 0.5, 1.0, 2.0]
nH = len(hScales)
baseH = getVariantRates(ratesDefault, CHECK_VARIANT)

ratesH = {k: np.array([v] * nH) for k, v in baseH.items() if k not in ("alphaPos", "alphaNeg")}
ratesH["alphaPos"] = np.array([scaleNodeH(baseH["alphaPos"], s) for s in hScales])
ratesH["alphaNeg"] = np.array([baseH["alphaNeg"]] * nH)

#Colors --> darker = stronger threshold
hColors = ["#c6dbef", "#9ecae1", "#4292c6", "#2171b5", "#084594"]

p3 = plotActivationFunctionVariants(ratesH, x2_fixed=x2Vals[0], x1Vals=x1Vals, w1=w1, w2=w2, X0=X0, t=t, labelBy="wh", colors=hColors)

#Save plot
#p3.output_backend = "svg"
#export_svgs(p3, filename="Simulation_Plots/mlp3/mlp3_hdrive_activation.svg")

In [ ]:
###Same sweep, as decision boundaries --> raising node h's drive should shrink the ON region
p4 = plotDecisionBoundaryVariants(ratesH, threshold=0.5, x1=x1, x2=x2, X1=X1, X2=X2, w1=w1, w2=w2, X0=X0, t=t, relative=True, labelBy="wh", colors=hColors)

#Save plot
#p4.output_backend = "svg"
#export_svgs(p4, filename="Simulation_Plots/mlp3/mlp3_hdrive_decision_boundary.svg")

In [ ]:
###Node-h ROTATION sweep, on the BASELINE weights --> no MLP2 equivalent, but see the caveat above.
###The threshold keeps its magnitude and only changes direction.
hAngles = [0.0, 30.0, 45.0, 60.0, 90.0]   #degrees from the X1 axis
nR = len(hAngles)
hMag = np.linalg.norm(baseH["alphaPos"][2])   #Keep the drive magnitude fixed


#rotateNodeH() --> Weight matrix with node h's row replaced by a fixed-magnitude vector at `ang`
def rotateNodeH(gateRows, mag, ang):
    h = mag * np.array([np.cos(np.deg2rad(ang)), np.sin(np.deg2rad(ang))])
    return np.vstack([np.asarray(gateRows, dtype=float)[:2], h])


ratesRot = {k: np.array([v] * nR) for k, v in baseH.items() if k not in ("alphaPos", "alphaNeg")}
ratesRot["alphaPos"] = np.array([rotateNodeH(baseH["alphaPos"], hMag, ang) for ang in hAngles])
ratesRot["alphaNeg"] = np.array([baseH["alphaNeg"]] * nR)

rotColors = ["#fdd0a2", "#fdae6b", "#f16913", "#d94801", "#7f2704"]

#EXPECT these boundaries to sit almost on top of each other. With j <- X1 and k <- X2 the ON region
#is a symmetric corner at (1,1), and near there x1 ~ x2, so every direction of w_h gives node h the
#same drive. This is the "rotation does nothing" case.
p5 = plotDecisionBoundaryVariants(ratesRot, threshold=0.5, x1=x1, x2=x2, X1=X1, X2=X2, w1=w1, w2=w2, X0=X0, t=t, relative=True, labelBy="wh", colors=rotColors)

#Save plot
#p5.output_backend = "svg"
#export_svgs(p5, filename="Simulation_Plots/mlp3/mlp3_hrotation_symmetric.svg")

In [ ]:
###Node-h ROTATION on an ASYMMETRIC ON region --> the case where direction actually buys something.
#Node j is driven broadly by BOTH inputs and node k needs X2, so the ON region is a horizontal BAND
#rather than a corner. w_h can now cut that band from the side (0 deg) or from the top (90 deg),
#which are genuinely different functions.
bandPos = np.array([[1.2, 1.2],     #node j --> broad, on for either input
                    [0.3, 1.8],     #node k --> needs X2
                    [0.0, 0.0]])    #node h --> replaced per angle by rotateNodeH()
bandNeg = np.array([[0.2, 0.2],     #node j
                    [0.6, 0.2],     #node k
                    [0.2, 0.2]])    #node h
bandMag = 1.4

ratesBand = {k: np.array([v] * nR) for k, v in baseH.items() if k not in ("alphaPos", "alphaNeg")}
ratesBand["alphaPos"] = np.array([rotateNodeH(bandPos, bandMag, ang) for ang in hAngles])
ratesBand["alphaNeg"] = np.array([bandNeg] * nR)

p6 = plotDecisionBoundaryVariants(ratesBand, threshold=0.5, x1=x1, x2=x2, X1=X1, X2=X2, w1=w1, w2=w2, X0=X0, t=t, relative=True, labelBy="wh", colors=rotColors)

#Save plot
#p6.output_backend = "svg"
#export_svgs(p6, filename="Simulation_Plots/mlp3/mlp3_hrotation_band.svg")

In [ ]:
###Quantifying the two rotation cases --> the claim in the markdown above, checked on the grid.
###symDiff = fraction of the input square whose ON/OFF label flips between two angles.
def onRegion(rates, gridX1, gridX2, frac=0.5):
    Y = np.zeros_like(gridX1)
    for i in range(gridX1.shape[1]):
        for j in range(gridX1.shape[0]):
            Y[j, i] = scipy.integrate.odeint(mlpODEs, X0, t, args=(w1 * gridX1[j, i], w2 * gridX2[j, i], rates))[-1, OUT]
    return Y, Y > frac * Y.max()


#A coarse grid is plenty for an area fraction, and keeps this cell quick
gq = np.linspace(0, 1, 21)
GQ1, GQ2 = np.meshgrid(gq, gq)

for label, rateSet in (("symmetric corner (baseline)", ratesRot), ("asymmetric band", ratesBand)):
    print(f"\n{label}:")
    masks = {}
    for vi, ang in enumerate(hAngles):
        Y, on = onRegion(getVariantRates(rateSet, vi), GQ1, GQ2)
        masks[ang] = on
        print(f"   w_h at {ang:>4g} deg:  max y_2 = {Y.max():6.3f}   ON-area = {on.mean():.3f}")
    sym = np.logical_xor(masks[hAngles[0]], masks[hAngles[-1]]).mean()
    print(f"   --> rotating {hAngles[0]:g} deg to {hAngles[-1]:g} deg flips {sym:.1%} of the input square")

In [ ]:
###Alpha comparison --> Variant 0 = baseline, Variants 1-2 scale alphaPos, Variants 3-4 scale alphaNeg
#Kinetics are held at the baseline values so only the weight matrices differ
nAlpha = 5
alphaBase = getVariantRates(ratesDefault, CHECK_VARIANT)
aPosBase = alphaBase["alphaPos"]
aNegBase = alphaBase["alphaNeg"]

ratesAlpha = {k: np.array([v] * nAlpha) for k, v in alphaBase.items() if k not in ("alphaPos", "alphaNeg")}
ratesAlpha["alphaPos"] = np.array([aPosBase, 0.5 * aPosBase, 0.25 * aPosBase, aPosBase, aPosBase])
ratesAlpha["alphaNeg"] = np.array([aNegBase, aNegBase, aNegBase, 0.25 * aNegBase, 0.5 * aNegBase])

#Colors --> Baseline black, alphaPos changes in reds, alphaNeg changes in blues
alphaColors = ["black", "#fc9272", "#de2d26", "#9ecae1", "#3182bd"]

p7 = plotDecisionBoundaryVariants(ratesAlpha, threshold=0.5, x1=x1, x2=x2, X1=X1, X2=X2, w1=w1, w2=w2, X0=X0, t=t, relative=True, labelBy="alpha", colors=alphaColors)

#Save plot
#p7.output_backend = "svg"
#export_svgs(p7, filename="Simulation_Plots/mlp3/mlp3_alpha_decision_boundary.svg")

In [ ]:
###plotHeatmap() --> Standalone heatmap of the steady-state output Y over the (X1, X2) grid for a single rate variant
def plotHeatmap(ratesDefault, variantIndex, x1, x2, X1, X2, w1, w2, X0, t, normal=True, labelBy="None"):
    rates = getVariantRates(ratesDefault, variantIndex)

    # Solve the ODEs over the full (X1, X2) grid to get steady-state output Y
    Y = np.zeros_like(X1)
    for i in range(len(x1)):
        for j in range(len(x2)):
            sol = scipy.integrate.odeint(mlpODEs, X0, t, args=(w1 * X1[j, i], w2 * X2[j, i], rates))
            Y[j, i] = sol[-1, OUT]  # Steady-state output y_2

    # Title --> Variant label (alpha, xi2, K_C2, K_Q2, wh) if given, otherwise the variant index
    label = getVariantLabel(rates, labelBy)
    title = label if label is not None else f"Heatmap"

    p_heatmap = figure(width=450, height=400, x_range=(0, 1), y_range=(0, 1), title=title)

    #Make the heatmap --> normal=True normalizes Y to [0, 1]
    contourf(
        p_heatmap,
        X1,
        X2,
        Y,
        palette=bokeh.palettes.Oranges8[::-1],
        normal=normal
    )

    p_heatmap.xaxis.axis_label = "X1 (Input plasmid 1)"
    p_heatmap.yaxis.axis_label = "X2 (Input plasmid 2)"

    show(p_heatmap)
    return p_heatmap

def heatmapForIllustrator(filename):
    with open(filename) as f:
        svg = f.read()
    # Illustrator only reads xlink:href for embedded images
    svg = re.sub(r'(<image[^>]*?)\shref=', r'\1 xlink:href=', svg)
    if 'xmlns:xlink' not in svg:
        svg = svg.replace('<svg ', '<svg xmlns:xlink="http://www.w3.org/1999/xlink" ', 1)
    with open(filename, "w") as f:
        f.write(svg)

In [ ]:
p8 = plotHeatmap(ratesDefault, variantIndex=0, x1=x1, x2=x2, X1=X1, X2=X2, w1=w1, w2=w2, X0=X0, t=t, normal=True, labelBy="None")
#p8.output_backend = "svg"
#for fn in export_svgs(p8, filename="Simulation_Plots/mlp3/mlp3_heatmap1.svg"):
#    heatmapForIllustrator(fn)

In [ ]:
p9 = plotHeatmap(ratesDefault, variantIndex=1, x1=x1, x2=x2, X1=X1, X2=X2, w1=w1, w2=w2, X0=X0, t=t, normal=True, labelBy="None")
#p9.output_backend = "svg"
#for fn in export_svgs(p9, filename="Simulation_Plots/mlp3/mlp3_heatmap2.svg"):
#    heatmapForIllustrator(fn)

In [ ]:
p10 = plotHeatmap(ratesDefault, variantIndex=2, x1=x1, x2=x2, X1=X1, X2=X2, w1=w1, w2=w2, X0=X0, t=t, normal=True, labelBy="None")
#p10.output_backend = "svg"
#for fn in export_svgs(p10, filename="Simulation_Plots/mlp3/mlp3_heatmap3.svg"):
#    heatmapForIllustrator(fn)

In [ ]:
p11 = plotHeatmap(ratesDefault, variantIndex=3, x1=x1, x2=x2, X1=X1, X2=X2, w1=w1, w2=w2, X0=X0, t=t, normal=True, labelBy="None")
#p11.output_backend = "svg"
#for fn in export_svgs(p11, filename="Simulation_Plots/mlp3/mlp3_heatmap4.svg"):
#    heatmapForIllustrator(fn)

In [ ]:
p12 = plotHeatmap(ratesDefault, variantIndex=4, x1=x1, x2=x2, X1=X1, X2=X2, w1=w1, w2=w2, X0=X0, t=t, normal=True, labelBy="None")
#p12.output_backend = "svg"
#for fn in export_svgs(p12, filename="Simulation_Plots/mlp3/mlp3_heatmap5.svg"):
#    heatmapForIllustrator(fn)

In [ ]:
#3D-Plot of the network activation function --> uses the Y grid from the last plotHeatmap call
#Recompute explicitly so this cell does not depend on loop leftovers
rates3D = getVariantRates(ratesDefault, CHECK_VARIANT)
Y3D = np.zeros_like(X1)
for i in range(len(x1)):
    for j in range(len(x2)):
        Y3D[j, i] = scipy.integrate.odeint(mlpODEs, X0, t, args=(w1 * X1[j, i], w2 * X2[j, i], rates3D))[-1, OUT]

fig = go.Figure(data=[
    go.Surface(z=Y3D, x=X1, y=X2, colorscale="Viridis")
])

fig.update_layout(
    scene=dict(
        xaxis_title='Input plasmid 1 (X1)',
        yaxis_title='Input plasmid 2 (X2)',
        zaxis_title='Network Output (Y2)'
    ),
    width=800,
    height=600
)

fig.show()

<u>Separability of the Network as Drawn</u>

Every production term that depends on an input is strictly proportional to an input plasmid, so
each hidden node's net drive carries no constant term:

$$\text{net}_n(x) = \sum_i \left(\alpha^{+}_{n,i} - \alpha^{-}_{n,i}\right) x_i$$

This means $\text{net}_n(0,0) = 0$ for all three hidden nodes, so **every hidden decision boundary
passes through the origin**. The hidden layer can rotate its three boundaries freely but cannot
translate them off $(0,0)$.

Adding node $h$ does not rescue this, and it is worth being precise about why, because $h$ is the
obvious candidate for a bias. Node $h$ is input-dependent where MLP2's $\theta$ was constant, which
is strictly more expressive -- but it is still **sign-constrained**. Its transcript only ever enters
the $b_2$ reaction, which *removes* $y^{(1)}_k$. So $h$ can subtract in an input-dependent way, but
it can never add. A negative, input-proportional term cannot lift $y^{(2)}(0,0)$ off zero.

And in fact nothing can here, for the same reason as MLP2: $\dot{c}^{(2)} \propto y^{(1)}_j y^{(1)}_k$
and both gate partners are zero at the origin, so the output is pinned to $0$ at $(0,0)$ by the
*product*.

__What MLP3 buys.__ The extra perceptron does not add a bias, but it does make the layer-2 threshold
a learned linear function of the inputs rather than a constant. The usable consequence is that the
ON region can be cut by a second, independently-oriented line -- so MLP3 realizes intersections of
half-planes through the origin that MLP2 cannot. The cell below measures this by comparing the
reachable margins of MLP3 against MLP2 restricted to the same kinetics.

The cell reports the **margin** $\min(\text{ON}) - \max(\text{OFF})$ for each target function --
positive means the function is realized, and the margin should be a decent fraction of the output
range to be usable. The search varies all three weight rows, since node $h$'s row is this
architecture's extra free knob.


In [ ]:
###Separability check --> the network exactly as drawn, no constitutive cassettes feeding the gate
ratesCheck = getVariantRates(ratesDefault, CHECK_VARIANT)


#truthTable() --> Steady-state output at the four corners of the input square
def truthTable(rates):
    return {(xa, xb): scipy.integrate.odeint(mlpODEs, X0, t, args=(w1 * xa, w2 * xb, rates))[-1, OUT]
            for xa in (0.0, 1.0) for xb in (0.0, 1.0)}


#margin() --> min(ON) - max(OFF). Positive means the target function is realized.
def margin(T, onCorners):
    on  = [T[c] for c in onCorners]
    off = [T[c] for c in T if c not in onCorners]
    return min(on) - max(off)


#withAlphas() --> Copy of a rates dict with the weight matrices swapped out
def withAlphas(rates, aPos, aNeg):
    swapped = dict(rates)
    swapped["alphaPos"] = aPos
    swapped["alphaNeg"] = aNeg
    return swapped


TARGETS = {
    "XOR": [(0.0, 1.0), (1.0, 0.0)],
    "OR":  [(0.0, 1.0), (1.0, 0.0), (1.0, 1.0)],
    "AND": [(1.0, 1.0)],
}

### The weights used throughout this notebook
T0 = truthTable(ratesCheck)
print("Network as configured above:")
print("  X1   X2  |   y_2")
for (xa, xb), y in sorted(T0.items()):
    print(f"  {xa:.1f}  {xb:.1f}  |  {y:.4f}")

outputRange = max(T0.values()) - min(T0.values())
netDrive = ratesCheck["alphaPos"] - ratesCheck["alphaNeg"]
print()
for n, name in enumerate(("j", "k", "h")):
    role = "gate partner" if name in ("j", "k") else "threshold (inhibitory only)"
    print(f"  net drive of node {name}:  {netDrive[n, 0]:+.2f} * X1  {netDrive[n, 1]:+.2f} * X2"
          f"   --> 0 at the origin   [{role}]")

### Can ANY weights reach each target? --> random search over the three alpha rows
#NOTE: a few hundred ODE solves of a 21-species system --> this takes a minute or two
rng = np.random.default_rng(0)
nTrials = {"XOR": 120, "OR": 50, "AND": 50}

#A margin must be judged on an ABSOLUTE scale. Scoring it as a fraction of the winning trial's
#own output range would call a dead all-zero network (margin 0 on range 0) a success, so the bar
#is a fixed fraction of the configured network's corner range instead.
scale = outputRange
print(f"\nBest margin found over random weight sets (bar = 10% of the configured range, {scale:.3g}):")
print("NOTE: alphas up to 3 make the system stiff --> an occasional ODEintWarning is expected here.")
for name, onCorners in TARGETS.items():
    #Seed the search with the weights this notebook is actually configured with. Without this the
    #random draw can miss a target the configured network already realizes (AND, for instance) and
    #the cell would report it unreachable, which would be flatly wrong.
    best = margin(T0, onCorners)
    bestRange = outputRange
    for _ in range(nTrials[name]):
        trialRates = withAlphas(ratesCheck, rng.uniform(0, 3, (3, 2)), rng.uniform(0, 3, (3, 2)))
        T = truthTable(trialRates)
        m = margin(T, onCorners)
        if m > best:
            best, bestRange = m, max(T.values()) - min(T.values())
    verdict = "reachable" if best > 0.1 * scale else "NOT reachable"
    #Reporting the winner's own range too --> a margin of 0 on a range of 0 is a dead network,
    #not a solution, and that is only visible when both numbers are shown
    print(f"  {name:<4} ({nTrials[name]:>3} trials):  margin {best:+.4f}  on its own range {bestRange:.4f}   --> {verdict}")

print("\nXOR still fails. All three hidden boundaries are pinned to the origin, and the output node")
print("multiplies y_j by y_k, so y_2(0,0) = 0 no matter how node h is weighted.")
print("Node h is an input-dependent threshold, not a bias --> it can only subtract.")